# Preprocesing pipeline for Sonics dataset for COLAB notebook.

### Getting all teh project from git hub

In [1]:
!rm -rf sample_data
!git clone https://github.com/thomas0barand/robust-deepfake-detector.git
!git checkout -b THOMAS/sonics
!cd robust-deepfake-detector/ && ls
!mv robust-deepfake-detector/* .
!rm -rf robust-deepfake-detector

Cloning into 'robust-deepfake-detector'...
remote: Enumerating objects: 1491, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 1491 (delta 69), reused 94 (delta 50), pack-reused 1324 (from 2)
Receiving objects: 100% (1491/1491), 1.22 GiB | 22.43 MiB/s, done.
Resolving deltas: 100% (590/590), done.
Updating files: 100% (102/102), done.
fatal: not a git repository (or any of the parent directories): .git
checkpoints  pyproject.toml  requirements.lock	scripts
data	     README.md	     research		src


In [2]:
!pip install nnAudio yt_dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.6 MB/s eta 0:00:00


### Imports + Variables

In [3]:
import os
import json
from pathlib import Path
from datetime import datetime
from huggingface_hub import snapshot_download
import kagglehub
import yt_dlp
import pandas as pd

import torch
import matplotlib.pyplot as plt

from nnAudio.features import STFT, CQT
from src.utils import load_audio, get_freqs, get_freqs_mask, get_spectrum, get_low_hull_curve

In [4]:
DATASET_DIR   = Path("/content/data/sonics")

## Download sonics

In [ ]:
!rm -rf /content/data/sonics/*

In [5]:
os.environ['KAGGLE_USERNAME'] = 'thomasb79'
os.environ['KAGGLE_KEY'] = 'KGAT_d19a42c61793db473d3aaf1499ccdeaa'

# Download latest version
path = kagglehub.dataset_download("awsaf49/sonics-dataset")

print("Path to dataset files:", path)

100%|██████████| 30.1G/30.1G [06:09<00:00, 87.3MB/s]


Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/awsaf49/sonics-dataset/versions/2


In [6]:
!mkdir /content/data/sonics/
!mv /root/.cache/kagglehub/datasets/awsaf49/sonics-dataset/versions/2/fake_songs /content/data/sonics/fake_songs
!mv /root/.cache/kagglehub/datasets/awsaf49/sonics-dataset/versions/2/fake_songs.csv /content/data/sonics/fake_songs.csv

In [7]:
!cd /content/data/sonics/fake_songs && ls -1 | wc -l

49074


**Download real_songs**

In [41]:
path = kagglehub.dataset_download("imsparsh/fma-free-music-archive-small-medium")

print("Path to dataset files:", path)

100%|██████████| 29.8G/29.8G [06:41<00:00, 79.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/imsparsh/fma-free-music-archive-small-medium/versions/1


In [42]:
os.makedirs("/content/data/sonics/real_songs", exist_ok=True)

!mv /root/.cache/kagglehub/datasets/imsparsh/fma-free-music-archive-small-medium/versions/1 /content/data/sonics/real_songs
!mv /content/data/sonics/real_songs/1/fma_metadata/tracks.csv /content/data/sonics/real_songs.csv
!mv /content/data/sonics/real_songs/1/fma_medium/fma_medium/*/* /content/data/sonics/real_songs/

!rm -rf /content/data/sonics/real_songs/1/

In [53]:
!cd /content/data/sonics/real_songs && ls -1 | wc -l

25000


## Fakeprint process

In [8]:
## process fakeprint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"working on {DEVICE}")

N_FFT = 1 << 14
SR = 16000
BINS_PER_OCTAVE = 192
AREA = 20
FREQ_RANGE = [1000, 8000]
FMIN = 32.7

hop_length = N_FFT // 2
fmax = SR / 2  # Maximum frequency that can be represented

working on cpu


## Split dataset

## Split SONICS

In [9]:
import pandas as pd
import numpy as np

split = pd.read_csv('/content/data/sonics/fake_songs.csv', low_memory=False)

s_train = split[split["split"]=="train"][["id", "source"]].to_numpy()
s_test = split[split["split"]=="test"][["id", "source"]].to_numpy()
s_valid = split[split["split"]=="valid"][["id", "source"]].to_numpy()

s_3_5 = split[split["algorithm"]=="chirp-v3.5"]["id"].to_numpy()
s_3 = split[split["algorithm"]=="chirp-v3"]["id"].to_numpy()
s_2 = split[split["algorithm"]=="chirp-v2-xxl-alpha"]["id"].to_numpy()
u_120 = split[split["algorithm"]=="udio-120s"]["id"].to_numpy()
u_30 = split[split["algorithm"]=="udio-30s"]["id"].to_numpy()

vsplit = {
    "suno_v3.5": [],
    "suno_v3": [],
    "suno_v2": [],
    "udio_v120": [],
    "udio_v30": [],
    "train": [],
    "test": [],
    "valid": [],
 }

for k, v in s_train:
    vsplit["train"].append( "fake_{:05d}_{}_0.mp3".format(k, v) )
    vsplit["train"].append( "fake_{:05d}_{}_1.mp3".format(k, v) )
for k, v in s_test:
    vsplit["test"].append( "fake_{:05d}_{}_0.mp3".format(k, v) )
    vsplit["test"].append( "fake_{:05d}_{}_1.mp3".format(k, v) )
for k, v in s_valid:
    vsplit["valid"].append( "fake_{:05d}_{}_0.mp3".format(k, v) )
    vsplit["valid"].append( "fake_{:05d}_{}_1.mp3".format(k, v) )

for k in s_3_5:
    vsplit["suno_v3.5"].append( "fake_{:05d}_suno_0.mp3".format(k) )
    vsplit["suno_v3.5"].append( "fake_{:05d}_suno_1.mp3".format(k) )
for k in s_3:
    vsplit["suno_v3"].append( "fake_{:05d}_suno_0.mp3".format(k) )
    vsplit["suno_v3"].append( "fake_{:05d}_suno_1.mp3".format(k) )
for k in s_2:
    vsplit["suno_v2"].append( "fake_{:05d}_suno_0.mp3".format(k) )
    vsplit["suno_v2"].append( "fake_{:05d}_suno_1.mp3".format(k) )
for k in u_120:
    vsplit["udio_v120"].append( "fake_{:05d}_udio_0.mp3".format(k) )
    vsplit["udio_v120"].append( "fake_{:05d}_udio_1.mp3".format(k) )
for k in u_30:
    vsplit["udio_v30"].append( "fake_{:05d}_udio_0.mp3".format(k) )
    vsplit["udio_v30"].append( "fake_{:05d}_udio_1.mp3".format(k) )


np.save("data/sonics/sonics_split.npy", vsplit)

## SPLIT FMA

In [55]:
import pandas as pd
import numpy as np
import shutil

# Load with multi-level header
tracks = pd.read_csv('/content/data/sonics/real_songs.csv',
                     index_col=0, header=[0, 1], low_memory=False)

# Check what split values exist
# medium = tracks[tracks[('set', 'subset')] == 'medium'or 'small' or 'large']


s_train = tracks[tracks[('set', 'split')] == 'training'].index.to_numpy()
s_valid = tracks[tracks[('set', 'split')] == 'validation'].index.to_numpy()
s_test  = tracks[tracks[('set', 'split')] == 'test'].index.to_numpy()

vsplit = {"train": [], "valid": [], "test": []}

for k in s_train:
    vsplit["train"].append("{:06d}.mp3".format(k))
for k in s_valid:
    vsplit["valid"].append("{:06d}.mp3".format(k))
for k in s_test:
    vsplit["test"].append("{:06d}.mp3".format(k))

for split_name, files in vsplit.items():
    print(f"{split_name}: {len(files)} files")

np.save("/content/data/sonics/fma_split.npy", vsplit)

# move to the correct folders

vsplit = np.load("/content/data/sonics/fma_split.npy", allow_pickle=True).item()

src_root = "/content/data/sonics/real_songs"

for split_name, files in vsplit.items():
    dst_dir = os.path.join(src_root, split_name)
    os.makedirs(dst_dir, exist_ok=True)

    not_found = 0
    for filename in files:
        src_path = os.path.join(src_root, filename)
        dst_path = os.path.join(dst_dir, filename)

        if os.path.exists(src_path):
            shutil.move(src_path, dst_path)
        else:
            not_found += 1

    print(f"{split_name}: {len(files) - not_found} moved, {not_found} not found")

train: 84353 files
valid: 10958 files
test: 11263 files
train: 19922 moved, 64431 not found
valid: 2505 moved, 8453 not found
test: 2573 moved, 8690 not found


In [56]:
# Rebuilt fma_split based on the remaining files
vsplit = {"train": [], "valid": [], "test": []}

for split_name in ["train", "valid", "test"]:
    split_dir = os.path.join(src_root, split_name)
    files = [f for f in os.listdir(split_dir) if f.endswith(".mp3")]
    vsplit[split_name] = files
    print(f"{split_name}: {len(files)} files")

np.save("/content/data/sonics/fma_split.npy", vsplit)

!cp /content/data/sonics/fma_split.npy "/content/drive/MyDrive/Robust deepfake detector/data/"

train: 19922 files
valid: 2505 files
test: 2573 files


# Processing fakeprints

In [10]:
from tqdm import tqdm, trange
from src.utils import load_audio, speed_up, get_spectrum, get_fakeprints
import soxr

def preprocess_fakeprints(
    file_paths,
    stft_transform,
    cqt_transform,
    batch_size=16,
    max_duration=60.0,
    attack_mode=None,
    bins_range=99,
    attack_range=None,
    n_fft=16384,
    sampling_rate=16000,
    bins_per_octave=192,
    hull_area=20,
    device=torch.device("cpu"),
):

    hop_length = n_fft // 2

    stft_transform = stft_transform.to(device)
    # cqt_transform = cqt_transform.to(device)

    files = []
    stft_fakeprints = []
    # cqt_fakeprints = []
    speed_factors = []

    num_batches = (len(file_paths) + batch_size - 1) // batch_size
    for i in range(num_batches) :#, leave=False, desc="Extracting fakeprints"):
        start = i * batch_size
        end = min((i + 1) * batch_size, len(file_paths))
        batch_files = file_paths[start:end]

        batch_waves = []
        for path in batch_files: #, leave=False, desc=f"Loading audio files for batch {i+1}/{num_batches}"):
            files.append(path)
            waveform, sr = load_audio(path, max_duration=max_duration)
            if waveform is None:
                continue

            if attack_mode == "discrete":
                bin_shifts = np.arange(-bins_range, bins_range)
                discrete_sf = 2 ** (bin_shifts / bins_per_octave)
                speed_factor = np.random.choice(discrete_sf) # Discrete speed factors for augmentation
                waveform = speed_up(waveform, sr, speed_factor)
                speed_factors.append(speed_factor)
            elif attack_mode == "continuous":
                speed_factor = np.random.uniform(attack_range[0], attack_range[1]) # Continuous speed factors for augmentation
                waveform = speed_up(waveform, sr, speed_factor)
                speed_factors.append(speed_factor)
            else:
                speed_factors.append(1.0)

            if sr != sampling_rate:
                waveform = soxr.resample(waveform.T, sr, sampling_rate, quality="VHQ").T
                waveform = torch.from_numpy(waveform)

            batch_waves.append(waveform)

        lengths = torch.tensor([w.shape[-1] for w in batch_waves], device=device)
        L_max = max(w.shape[-1] for w in batch_waves)

        padded = torch.zeros(len(batch_waves), 1, L_max)
        for k, w in enumerate(batch_waves):
            padded[k, :, :w.shape[-1]] = w
        padded = padded.to(device) # (B, 1, Lmax)

        stft_batch = get_spectrum(stft_transform, padded) # (B, n_bins, T')
        # cqt_batch = get_spectrum(cqt_transform, padded) # (B, n_bins, T')

        T_frames = stft_batch.shape[-1]
        frame_lengths = ((lengths - n_fft) // hop_length + 1).unsqueeze(1) # (B, 1)
        mask = torch.arange(T_frames, device=device).unsqueeze(0) < frame_lengths # (B, T')

        # Average over time dimension, accounting for varying lengths
        stft_batch = (stft_batch * mask.unsqueeze(1)).sum(-1) / frame_lengths.float() # (B, n_bins)
        # cqt_batch = (cqt_batch * mask.unsqueeze(1)).sum(-1) / frame_lengths.float() # (B, n_bins)

        stft_fp = get_fakeprints(stft_batch, area=hull_area)
        # cqt_fp = get_fakeprints(cqt_batch, area=hull_area)

        stft_fakeprints.append(stft_fp)
        # cqt_fakeprints.append(cqt_fp)

    stft_fakeprints = torch.cat(stft_fakeprints, dim=0)
    # cqt_fakeprints = torch.cat(cqt_fakeprints, dim=0)
    speed_factors = np.array(speed_factors)

    return {
        "files": files,
        "stft": stft_fakeprints.cpu().numpy(),
        #"cqt": cqt_fakeprints.cpu().numpy(),
        "speed_factors": speed_factors,
        "n_fft": n_fft,
        "sampling_rate": sampling_rate,
        "bins_per_octave": bins_per_octave,
        "hull_area": hull_area,
    }

In [11]:
import librosa

def is_valid_mp3(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, duration=1.0)  # Try loading a short segment to check validity
        return True
    except Exception as e:
        print(f"Error opening {file_path}: {e}")
        return False

def pipeline(
    file_paths,
    out_dir,
    batch_size=16,
    max_duration=60.0,
    attack_mode=None,
    bins_range=99,
    attack_range=None,
    shard_size=500,
    shard_start=0,
    num_shards=None,
    n_fft=16384,
    sampling_rate=16000,
    bins_per_octave=192,
    hull_area=20,
    fmin=32.7,
    device=torch.device("cpu"),
):
    os.makedirs(out_dir, exist_ok=True)
    file_paths = sorted([p for p in file_paths if is_valid_mp3(p)])
    shards = [file_paths[i:i+shard_size] for i in range(0, len(file_paths), shard_size)]
    print(f"Total files: {len(file_paths)}, Shards: {len(shards)}")

    hop_length = n_fft // 2
    fmax = sampling_rate / 2  # Maximum frequency that can be represented

    stft_transform = STFT(
        n_fft=n_fft,
        sr=sampling_rate,
        hop_length=hop_length,
        fmin=fmin,
        fmax=fmax,
        output_format="Magnitude",
        verbose=False,
    )

    cqt_transform = CQT(
        sr=sampling_rate,
        hop_length=hop_length,
        fmin=fmin,
        fmax=fmax,
        bins_per_octave=bins_per_octave,
        output_format="Magnitude",
        verbose=False,
    )

    end = len(shards) if not num_shards else min(shard_start + num_shards, len(shards))
    for i in trange(shard_start, end):
        shard_paths = shards[i]
        shard = preprocess_fakeprints(
            shard_paths,
            stft_transform=stft_transform,
            cqt_transform=cqt_transform,
            batch_size=batch_size,
            max_duration=max_duration,
            attack_mode=attack_mode,
            bins_range=bins_range,
            attack_range=attack_range,
            n_fft=n_fft,
            sampling_rate=sampling_rate,
            bins_per_octave=bins_per_octave,
            hull_area=hull_area,
            device=device,
        )
        np.savez(f"{out_dir}/fakeprints_{i+1:02d}.npz", **shard)
        # os.system(f"cp -r {OUT_DIR} '/content/drive/MyDrive/Robust deepfake detector/data/sonics/attack/'{split}/")


In [12]:
label = "fake_songs/"
split = "test"

if label == "fake_songs/":
  vsplit = np.load("data/sonics/sonics_split.npy", allow_pickle=True).item()
  gen_model = 'suno_v3.5'
  files = list(set(vsplit[gen_model]) & set(vsplit[split]))
  file_paths = ["/content/data/sonics/" + label + file for file in files]
elif label == "real_songs/":
  gen_model = 'human'
  vsplit = np.load("data/sonics/fma_split.npy", allow_pickle=True).item()
  files = list(vsplit[split])
  file_paths = ["/content/data/sonics/" + label + split + "/" + file for file in files]

print(f"{len(file_paths)} files in '{split}' split")
print(file_paths[:3])


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"working on {DEVICE}")

N_FFT = 1 << 14
SR = 16000
BINS_PER_OCTAVE = 192
AREA = 20
FREQ_RANGE = [1000, 8000]
FMIN = 32.7

hop_length = N_FFT // 2
fmax = SR / 2  # Maximum frequency that can be represented

ATTACK_MODE = None
ATTACK_RANGE = [0.7, 1.4]

OUT_DIR = Path("data", "fakeprints", "attack", "discrete", split, gen_model)
print(OUT_DIR)

bin_low  = BINS_PER_OCTAVE * np.log2(ATTACK_RANGE[0])
bin_high = BINS_PER_OCTAVE * np.log2(ATTACK_RANGE[1])

# Symetric RANGE of bins
BINS_RANGE = int(np.ceil(max(abs(bin_low), abs(bin_high))))
print(f"BINS_RANGE: {BINS_RANGE}")

1824 files in 'test' split
['/content/data/sonics/fake_songs/fake_32042_suno_0.mp3', '/content/data/sonics/fake_songs/fake_43107_suno_0.mp3', '/content/data/sonics/fake_songs/fake_24486_suno_1.mp3']
working on cpu
data/fakeprints/attack/discrete/test/suno_v3.5
BINS_RANGE: 99


In [13]:
np.random.seed(42)
torch.manual_seed(42)

os.makedirs(f'/content/drive/MyDrive/Robust deepfake detector/data/sonics/attack/{split}/{gen_model}/', exist_ok=True)

pipeline(
        file_paths,
        out_dir=OUT_DIR,
        batch_size=16,
        max_duration=30,
        attack_mode=ATTACK_MODE,
        bins_range=BINS_RANGE,
        attack_range=ATTACK_RANGE,
        shard_size=500,
        shard_start=0,
        num_shards=None,
        n_fft=N_FFT,
        sampling_rate=SR,
        bins_per_octave=BINS_PER_OCTAVE,
        hull_area=AREA,
        fmin=FMIN,
        device=DEVICE,
    )

/tmp/ipykernel_1683/1145081060.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None, duration=1.0)  # Try loading a short segment to check validity
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error opening /content/data/sonics/fake_songs/fake_28886_suno_1.mp3: [Errno 2] No such file or directory: '/content/data/sonics/fake_songs/fake_28886_suno_1.mp3'
Error opening /content/data/sonics/fake_songs/fake_36502_suno_1.mp3: [Errno 2] No such file or directory: '/content/data/sonics/fake_songs/fake_36502_suno_1.mp3'
Error opening /content/data/sonics/fake_songs/fake_16161_suno_1.mp3: [Errno 2] No such file or directory: '/content/data/sonics/fake_songs/fake_16161_suno_1.mp3'
Total files: 1821, Shards: 4


/usr/local/lib/python3.12/dist-packages/nnAudio/utils.py:429: SyntaxWarning: If fmax is given, n_bins will be ignored
  warnings.warn("If fmax is given, n_bins will be ignored", SyntaxWarning)
100%|██████████| 4/4 [50:31<00:00, 757.82s/it]


### LOAD .NPZ FAKEPRINTS

In [ ]:
fp = np.load("/content/data/sonics/fakeprints/suno_v3.5/valid/fakeprints_01.npz")
print(fp["stft"].shape)

(500, 8193)
